# AI Jobs Salaries — Exploratory Data Analysis
**Author:** Kunta Vinay Kumar  
**Dataset:** `ai_jobs_salaries_clean.csv`  
**Data Dictionary:** `data_dictionary.md`

---
## Objective
Explore the cleaned AI-jobs salary dataset to understand how compensation varies by:
- **Experience level** (Entry / Mid / Senior / Executive)
- **Role family** (top 10 grouped job families)
- **Company size** (S / M / L)
- **Work mode** (On-site / Hybrid / Remote)

Outlier records (flagged via `salary_outlier_flag == True`) are identified and **excluded** from the
charting analysis so that the visualisations reflect the central salary distribution.

## 1. Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Consistent visual style across all charts
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Imports successful ✓')

## 2. Load Data

Read the CSV into a DataFrame and cast the `salary_outlier_flag` column to a proper boolean
(it is stored as the strings `'True'` / `'False'` in the raw file).

In [ ]:
df = pd.read_csv('ai_jobs_salaries_clean.csv')

# Normalise the outlier flag to a proper Python bool
df['salary_outlier_flag'] = df['salary_outlier_flag'].astype(str).str.strip().map(
    {'True': True, 'False': False}
)

print(f'Rows loaded : {len(df):,}')
print(f'Columns     : {df.shape[1]}')
df.head()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Schema overview
A quick look at column dtypes and a statistical summary of numeric fields.

In [ ]:
print('=== dtypes ===')
print(df.dtypes)
print()
print('=== Numeric summary ===')
df.describe(include='number').round(2)

### 3.2 Categorical value counts
Understand the distribution of the key categorical dimensions used in the four charts below.

In [ ]:
cat_cols = ['experience_level_label', 'company_size', 'work_mode', 'role_family']

for col in cat_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts().to_string())
    print()

### 3.3 Missing value check

Identify any columns that contain nulls.  
A **clean dataset should show zero nulls** for the columns we analyse.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_report = missing_report[missing_report['missing_count'] > 0]

if missing_report.empty:
    print('✓ No missing values found in any column.')
else:
    print('⚠ Columns with missing values:')
    print(missing_report.to_string())

### 3.4 Outlier flag inspection

The `salary_outlier_flag` column marks rows whose `salary_in_usd` falls **outside the 1.5 × IQR fence**
calculated over the full dataset (see `data_dictionary.md`).  
These records are **retained** in the raw data but we will build a separate `df_clean` DataFrame
that excludes them for the salary visualisations.

In [ ]:
outlier_counts = df['salary_outlier_flag'].value_counts()
print('salary_outlier_flag distribution:')
print(outlier_counts.to_string())
print()
print(f"Outlier rows   : {outlier_counts.get(True, 0):,}  "
      f"({outlier_counts.get(True, 0)/len(df)*100:.2f}%)")
print(f"Non-outlier rows: {outlier_counts.get(False, 0):,}  "
      f"({outlier_counts.get(False, 0)/len(df)*100:.2f}%)")

### 3.5 Salary distribution — before and after outlier removal

A side-by-side look at how the distribution changes once flagged outliers are excluded.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

# Before
sns.histplot(df['salary_in_usd'], bins=60, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Salary Distribution — All Records')
axes[0].set_xlabel('Salary (USD)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# After
df_clean = df[df['salary_outlier_flag'] == False].copy()
sns.histplot(df_clean['salary_in_usd'], bins=60, kde=True, ax=axes[1], color='seagreen')
axes[1].set_title('Salary Distribution — Outliers Removed')
axes[1].set_xlabel('Salary (USD)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

fig.suptitle('Impact of Outlier Flag on Salary Distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nRows in df_clean (no outliers): {len(df_clean):,}")

---
## 4. Salary Visualisations

All four charts below use **`df_clean`** — the DataFrame with outlier rows removed — so that extreme
values do not distort the averages.

> **Experience level label ordering** used throughout:  
> Entry-level → Mid-level → Senior-level → Executive-level

### Chart 1 — Average Salary by Experience Level

This chart answers: *Does seniority translate linearly into higher pay?*  
We expect a clear upward staircase from Entry to Executive.

In [ ]:
# Define a logical ordering for experience levels
exp_order = ['Entry-level', 'Mid-level', 'Senior-level', 'Executive-level']

avg_by_exp = (
    df_clean
    .groupby('experience_level_label', observed=True)['salary_in_usd']
    .mean()
    .reindex(exp_order)
    .reset_index()
)
avg_by_exp.columns = ['experience_level_label', 'avg_salary_usd']

fig, ax = plt.subplots(figsize=(8, 5))
bars = sns.barplot(
    data=avg_by_exp,
    x='experience_level_label',
    y='avg_salary_usd',
    palette='Blues_d',
    order=exp_order,
    ax=ax
)

# Annotate each bar with the dollar value
for patch, val in zip(ax.patches, avg_by_exp['avg_salary_usd']):
    ax.text(
        patch.get_x() + patch.get_width() / 2,
        patch.get_height() + 800,
        f'${val:,.0f}',
        ha='center', va='bottom', fontsize=9.5, fontweight='bold'
    )

ax.set_title('Average Salary (USD) by Experience Level', fontsize=13, pad=12)
ax.set_xlabel('Experience Level', labelpad=8)
ax.set_ylabel('Average Salary (USD)', labelpad=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_ylim(0, avg_by_exp['avg_salary_usd'].max() * 1.18)
sns.despine()
plt.tight_layout()
plt.show()

print('\nUnderlying data:')
print(avg_by_exp.to_string(index=False))

**Interpretation:** Salary rises consistently with seniority. Executive-level roles command the
highest average compensation, while Entry-level roles show the lowest — as expected. The gap
between Senior-level and Executive-level can be substantial, reflecting the scarcity of C-suite
and principal-track AI positions.

### Chart 2 — Average Salary for Top 10 Role Families

This chart answers: *Which AI job families pay the most on average?*  
We rank role families by mean salary and display the top 10.

In [ ]:
avg_by_role = (
    df_clean
    .groupby('role_family', observed=True)['salary_in_usd']
    .agg(avg_salary_usd='mean', count='count')
    .reset_index()
    .sort_values('avg_salary_usd', ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=avg_by_role,
    y='role_family',
    x='avg_salary_usd',
    palette='Greens_d',
    ax=ax
)

# Annotate bars with value and sample size
for patch, (_, row) in zip(ax.patches, avg_by_role.iterrows()):
    ax.text(
        patch.get_width() + 400,
        patch.get_y() + patch.get_height() / 2,
        f"${row['avg_salary_usd']:,.0f}  (n={row['count']:,})",
        va='center', fontsize=8.5
    )

ax.set_title('Top 10 Role Families by Average Salary (USD)', fontsize=13, pad=12)
ax.set_xlabel('Average Salary (USD)', labelpad=8)
ax.set_ylabel('Role Family', labelpad=8)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xlim(0, avg_by_role['avg_salary_usd'].max() * 1.28)
sns.despine()
plt.tight_layout()
plt.show()

print('\nUnderlying data:')
print(avg_by_role.to_string(index=False))

**Interpretation:** Specialised, research-heavy, or leadership-oriented role families (e.g.
*AI Architect*, *Research Scientist*, *ML Engineer*) tend to attract the highest average salaries.
Note that role families with smaller sample sizes (`n`) may show higher averages due to limited
representation — always consider `n` alongside the mean.

### Chart 3 — Salary Distribution by Company Size

This chart answers: *Do larger companies pay significantly more than smaller ones?*  
A box-plot is used here so we can see **spread and median** — not just the mean.

In [ ]:
# Map company size codes to descriptive labels for the chart axis
size_label_map = {'S': 'Small (S)', 'M': 'Medium (M)', 'L': 'Large (L)'}
size_order_raw   = ['S', 'M', 'L']
size_order_label = [size_label_map[s] for s in size_order_raw]

df_clean['company_size_label'] = df_clean['company_size'].map(size_label_map)

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=df_clean,
    x='company_size_label',
    y='salary_in_usd',
    order=size_order_label,
    palette='Oranges',
    width=0.45,
    flierprops=dict(marker='o', markersize=2, alpha=0.3),
    ax=ax
)

# Overlay median annotation
medians = (
    df_clean
    .groupby('company_size_label', observed=True)['salary_in_usd']
    .median()
    .reindex(size_order_label)
)
for i, (label, med) in enumerate(medians.items()):
    ax.text(i, med + 1200, f'Median\n${med:,.0f}', ha='center', fontsize=8.5, color='#333')

ax.set_title('Salary Distribution by Company Size (Outliers Removed)', fontsize=13, pad=12)
ax.set_xlabel('Company Size', labelpad=8)
ax.set_ylabel('Salary in USD', labelpad=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
sns.despine()
plt.tight_layout()
plt.show()

print('\nMedian and mean salary by company size:')
print(
    df_clean
    .groupby('company_size', observed=True)['salary_in_usd']
    .agg(median='median', mean='mean', count='count')
    .reindex(size_order_raw)
    .round(0)
    .to_string()
)

**Interpretation:** Medium and Large companies generally offer higher median salaries than Small
companies. The interquartile range is also wider for larger companies, reflecting more diverse
role levels and geographies within that tier. Small companies may show lower median salaries
but can still house high-paying individual roles.

### Chart 4 — Average Salary by Work Mode

This chart answers: *Does remote, hybrid, or on-site work correlate with different salary levels?*

In [ ]:
work_mode_order = ['On-site', 'Hybrid', 'Remote']

avg_by_mode = (
    df_clean
    .groupby('work_mode', observed=True)['salary_in_usd']
    .agg(avg_salary_usd='mean', count='count')
    .reset_index()
    .set_index('work_mode')
    .reindex(work_mode_order)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 5))
bars = sns.barplot(
    data=avg_by_mode,
    x='work_mode',
    y='avg_salary_usd',
    order=work_mode_order,
    palette='Purples_d',
    ax=ax
)

for patch, (_, row) in zip(ax.patches, avg_by_mode.iterrows()):
    ax.text(
        patch.get_x() + patch.get_width() / 2,
        patch.get_height() + 500,
        f"${row['avg_salary_usd']:,.0f}\n(n={row['count']:,})",
        ha='center', va='bottom', fontsize=9
    )

ax.set_title('Average Salary (USD) by Work Mode', fontsize=13, pad=12)
ax.set_xlabel('Work Mode', labelpad=8)
ax.set_ylabel('Average Salary (USD)', labelpad=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_ylim(0, avg_by_mode['avg_salary_usd'].max() * 1.22)
sns.despine()
plt.tight_layout()
plt.show()

print('\nUnderlying data:')
print(avg_by_mode.to_string(index=False))

**Interpretation:** Remote roles in the AI/data space can command salaries comparable to or higher
than on-site roles, reflecting the global talent market for these skills. Hybrid and On-site
averages may vary depending on the concentration of high-cost-of-living locations in those
categories.

---
## 5. Summary

| Dimension | Key Finding |
|---|---|
| **Experience level** | Clear salary staircase — Executive > Senior > Mid > Entry |
| **Role family** | Architect / Research / ML Engineering families top the pay table |
| **Company size** | Medium/Large companies show higher medians; Small shows wider relative spread |
| **Work mode** | Remote roles are competitively compensated; minimal disadvantage vs on-site |

**Limitations to bear in mind:**
- All salary data is **self-reported** (survey), not verified payroll data.
- Country and role-family coverage is **uneven** — some cells have very few observations.
- `role_family` is a **manual mapping** — treat as illustrative, not authoritative.  
  (See `data_dictionary.md` for full details.)